In [1]:
"""
PASSO 6 — Industry_Alg: Matrizes Setor × Algoritmo.

Replica _IM_Industry_Alg.R:
    Lê os arquivos .Performance_{Algo}_{Setor}_{Sample}_{Cenário}.csv
    gerados pelo Eva_Performance e monta 8 matrizes (uma por métrica)
    com linhas = [Index, BAH, LR, SVM, ..., eNFN] e colunas = setores.

    Para cada setor:
    - Index e BAH: médias do índice e buy-and-hold (extraídas do 1º algoritmo)
    - Cada algoritmo: média da métrica "Trade" naquele setor

Saída: {MÉTRICA}_{Sample}_{Período}.csv
    Exemplo: ASR_In_InS.csv → Sharpe anualizado, ações In-Sample, período treino

    Total: 8 métricas × 4 cenários = 32 arquivos

Uso:
    python 06_industry_alg.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

ALGORITHMS = ["LR", "SVM", "NB", "CART", "RF", "XGB",
              "MLP", "DBN", "SAE", "RNN", "LSTM", "GRU"]

# Mapeamento cenário → (Sample, Período)
SCENARIOS = {
    "In_InS":   {"Sample": "In",  "Period": "InS"},
    "In_OutS":  {"Sample": "In",  "Period": "OutS"},
    "Out_InS":  {"Sample": "Out", "Period": "InS"},
    "Out_OutS": {"Sample": "Out", "Period": "OutS"},
}

# As 8 métricas na ordem do R
# Cada métrica tem 3 colunas no Performance: {Metric}Index, {Metric}BAH, {Metric}Trade
METRICS = ["WR", "ARR", "ASR", "MDD", "CAR", "SOR", "MSR", "ART"]

# Nome das linhas: Index, BAH, depois os 13 algoritmos
ROW_NAMES = ["Index", "BAH"] + ALGORITHMS
# ========================================================


def build_matrices(base_dir: Path, sec_names_path: Path):
    """Constrói as 32 matrizes (8 métricas × 4 cenários)."""

    # Ler setores disponíveis
    codes_df = pd.read_csv(sec_names_path, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    industries = sorted(codes_df["industry"].str.strip().unique())

    print(f"Algoritmos: {len(ALGORITHMS)} ({ALGORITHMS})")
    print(f"Setores: {industries}")
    print(f"Cenários: {list(SCENARIOS.keys())}")
    print(f"Métricas: {METRICS}")
    print(f"Total de matrizes: {len(METRICS) * len(SCENARIOS)}\n")

    n_rows = len(ROW_NAMES)     # Index + BAH + 13 algoritmos = 15
    n_cols = len(industries)

    for scen_name, scen_params in SCENARIOS.items():
        sample = scen_params["Sample"]
        period = scen_params["Period"]

        # Inicializar 8 matrizes vazias
        matrices = {m: np.full((n_rows, n_cols), np.nan) for m in METRICS}

        for j, ind in enumerate(industries):
            index_bah_filled = False

            for algo_idx, algo in enumerate(ALGORITHMS):
                # Arquivo de performance gerado pelo Eva_Performance
                perf_file = base_dir / f".Performance_{algo}_{ind}_{sample}_{scen_name}.csv"

                if not perf_file.exists():
                    continue

                try:
                    perf = pd.read_csv(perf_file, index_col=0, encoding="utf-8-sig")
                except Exception:
                    continue

                if perf.empty:
                    continue

                # Calcular médias por coluna
                means = perf.mean(axis=0)

                # Na primeira vez para este setor, preencher Index e BAH
                if not index_bah_filled:
                    for m_idx, metric in enumerate(METRICS):
                        col_index = f"{metric}Index"
                        col_bah = f"{metric}BAH"

                        if col_index in means.index:
                            matrices[metric][0, j] = means[col_index]  # Index (linha 0)
                        if col_bah in means.index:
                            matrices[metric][1, j] = means[col_bah]    # BAH (linha 1)

                    index_bah_filled = True

                # Preencher linha do algoritmo (Trade)
                row_idx = algo_idx + 2  # +2 porque Index=0, BAH=1
                for m_idx, metric in enumerate(METRICS):
                    col_trade = f"{metric}Trade"
                    if col_trade in means.index:
                        matrices[metric][row_idx, j] = means[col_trade]

        # Salvar as 8 matrizes para este cenário
        for metric in METRICS:
            # Mapear nome do cenário para formato do R:
            # "In_InS" → Sample="In", Period="InS"
            outfile = base_dir / f"{metric}_{sample}_{period}.csv"

            df_out = pd.DataFrame(
                matrices[metric],
                index=ROW_NAMES,
                columns=industries,
            )
            df_out.index.name = ""
            df_out.to_csv(outfile, encoding="utf-8-sig")

        print(f"  Cenário {scen_name}: 8 matrizes salvas")

    print(f"\n{'='*50}")
    print(f"Concluído! {len(METRICS) * len(SCENARIOS)} matrizes geradas.")
    print(f"Formato: {{MÉTRICA}}_{{Sample}}_{{Período}}.csv")
    print(f"Linhas: {ROW_NAMES}")
    print(f"Colunas: {industries}")


if __name__ == "__main__":
    build_matrices(BASE_DIR, SEC_NAMES)

Algoritmos: 12 (['LR', 'SVM', 'NB', 'CART', 'RF', 'XGB', 'MLP', 'DBN', 'SAE', 'RNN', 'LSTM', 'GRU'])
Setores: ['BM', 'CC', 'CNC', 'COM', 'FIN', 'GCS', 'OGB', 'UTI']
Cenários: ['In_InS', 'In_OutS', 'Out_InS', 'Out_OutS']
Métricas: ['WR', 'ARR', 'ASR', 'MDD', 'CAR', 'SOR', 'MSR', 'ART']
Total de matrizes: 32

  Cenário In_InS: 8 matrizes salvas
  Cenário In_OutS: 8 matrizes salvas
  Cenário Out_InS: 8 matrizes salvas
  Cenário Out_OutS: 8 matrizes salvas

Concluído! 32 matrizes geradas.
Formato: {MÉTRICA}_{Sample}_{Período}.csv
Linhas: ['Index', 'BAH', 'LR', 'SVM', 'NB', 'CART', 'RF', 'XGB', 'MLP', 'DBN', 'SAE', 'RNN', 'LSTM', 'GRU']
Colunas: ['BM', 'CC', 'CNC', 'COM', 'FIN', 'GCS', 'OGB', 'UTI']
